In [1]:
# ==========================================
# CÀI ĐẶT VÀ TRÍCH XUẤT XÁC SUẤT TỪ DỮ LIỆU
# ==========================================
import numpy as np
import pandas as pd

print("Đang tải dữ liệu 'E-commerce Sales History' từ Kaggle...")
# Giả lập thao tác load bộ dữ liệu bán hàng lịch sử trong 100 ngày
np.random.seed(42)
# Khách hàng có xu hướng mua từ 0 đến 5 sản phẩm/ngày
historical_demand = np.random.choice([0, 1, 2, 3, 4, 5], size=100, p=[0.1, 0.3, 0.3, 0.15, 0.1, 0.05])

df_sales = pd.DataFrame({'Day': range(1, 101), 'Items_Sold': historical_demand})
display(df_sales.head())

# Tính toán phân phối xác suất nhu cầu (Demand Probability) từ dữ liệu
demand_counts = df_sales['Items_Sold'].value_counts(normalize=True).sort_index()
prob_demand = demand_counts.to_dict()
print(f"\nXác suất nhu cầu mua hàng của khách (từ Data): \n{prob_demand}\n")

# ==========================================
# BƯỚC 1: ĐỊNH NGHĨA MDP CHO KHO HÀNG
# ==========================================
MAX_CAPACITY = 5 # Sức chứa tối đa của kho là 5 sản phẩm
states = list(range(MAX_CAPACITY + 1)) # S = {0, 1, 2, 3, 4, 5}

# Các tham số tài chính (Bài toán Kinh tế)
PRICE_PER_ITEM = 100.0    # Giá bán ra: 100k
COST_PER_ITEM = 40.0      # Giá vốn nhập hàng: 40k
STORAGE_COST = 5.0        # Phí lưu trữ qua đêm cho mỗi sản phẩm: 5k/đêm
PENALTY_OUT_OF_STOCK = 20.0 # Phạt mất khách nếu khách hỏi mà hết hàng: 20k

gamma = 0.95 # Hệ số chiết khấu gamma (0.95: Ưu tiên lợi nhuận dài hạn)
epsilon = 0.01 # Ngưỡng dừng của hội tụ

# Hàm tính phần thưởng kỳ vọng R(s, a) và xác suất chuyển đổi
def calculate_transitions_and_rewards(s, a):
    expected_reward = 0
    next_state_probs = {s_prime: 0 for s_prime in states}
    
    # Hàng trong kho sáng hôm sau = Tồn kho cũ (s) + Hàng mới nhập (a)
    stock_morning = s + a 
    
    # Trừ ngay chi phí nhập hàng
    expected_reward -= (a * COST_PER_ITEM)
    
    # Duyệt qua các khả năng khách mua hàng (Demand)
    for demand, prob in prob_demand.items():
        # Số lượng bán được thực tế (không thể bán nhiều hơn số hàng có sẵn)
        sold = min(stock_morning, demand)
        
        # Doanh thu bán hàng
        revenue = sold * PRICE_PER_ITEM
        
        # Hàng dư cuối ngày
        stock_end_of_day = stock_morning - sold
        
        # Nếu khách muốn mua nhiều hơn số hàng trong kho -> Bị phạt Out-of-Stock
        penalty = 0
        if demand > stock_morning:
            penalty = (demand - stock_morning) * PENALTY_OUT_OF_STOCK
            
        # Phí lưu kho qua đêm cho hàng ế
        storage = stock_end_of_day * STORAGE_COST
        
        # Cập nhật tổng phần thưởng kỳ vọng (Reward)
        step_reward = revenue - penalty - storage
        expected_reward += prob * step_reward
        
        # Cập nhật xác suất cho trạng thái ngày hôm sau
        next_state_probs[stock_end_of_day] += prob
        
    return expected_reward, next_state_probs

# ==========================================
# BƯỚC 2: THUẬT TOÁN VALUE ITERATION
# ==========================================
U = {s: 0.0 for s in states} # Khởi tạo Tiện ích bằng 0

iteration = 0
while True:
    delta = 0
    U_new = U.copy()
    
    for s in states:
        max_utility = float('-inf')
        
        # Hành động: Chỉ có thể nhập thêm sao cho tổng hàng không vượt MAX_CAPACITY
        possible_actions = list(range(MAX_CAPACITY - s + 1))
        
        for a in possible_actions:
            # Lấy Phần thưởng kỳ vọng và Xác suất chuyển đổi
            R_sa, P_next = calculate_transitions_and_rewards(s, a)
            
            # Bellman Equation: U(s) = max_a [ R(s,a) + gamma * sum(P(s'|s,a) * U(s')) ]
            expected_future_utility = sum(prob * U[s_prime] for s_prime, prob in P_next.items())
            total_utility = R_sa + gamma * expected_future_utility
            
            if total_utility > max_utility:
                max_utility = total_utility
                
        U_new[s] = max_utility
        delta = max(delta, abs(U_new[s] - U[s]))
        
    U = U_new
    iteration += 1
    
    if delta < epsilon:
        print(f"✅ Value Iteration hội tụ sau {iteration} vòng lặp.\n")
        break

# ==========================================
# BƯỚC 3: TRÍCH XUẤT CHÍNH SÁCH NHẬP HÀNG (POLICY)
# ==========================================
optimal_policy = {}

for s in states:
    best_action = None
    max_utility = float('-inf')
    possible_actions = list(range(MAX_CAPACITY - s + 1))
    
    for a in possible_actions:
        R_sa, P_next = calculate_transitions_and_rewards(s, a)
        utility = R_sa + gamma * sum(prob * U[s_prime] for s_prime, prob in P_next.items())
        
        if utility > max_utility:
            max_utility = utility
            best_action = a
            
    optimal_policy[s] = best_action

# In ra sách hướng dẫn vận hành kho (Chính sách)
print("📊 BẢNG GIÁ TRỊ TIỆN ÍCH (EXPECTED UTILITY) TẠI MỖI TRẠNG THÁI:")
for s, u in U.items():
    print(f"- Tồn kho {s} sản phẩm: {u:.2f} (Giá trị kỳ vọng lâu dài)")

print("\n🚀 CHÍNH SÁCH NHẬP HÀNG TỐI ƯU (OPTIMAL POLICY π*):")
print("-" * 50)
for s, a in optimal_policy.items():
    if a == 0:
        print(f"Kho đang có [ {s} ] món  ->  Không cần nhập thêm.")
    else:
        print(f"Kho đang có [ {s} ] món  ->  Đặt nhập thêm [ {a} ] món.")

Đang tải dữ liệu 'E-commerce Sales History' từ Kaggle...


,Day,Items_Sold
0,1,1
1,2,5
2,3,3
3,4,2
4,5,1



Xác suất nhu cầu mua hàng của khách (từ Data): 
{0: 0.13, 1: 0.33, 2: 0.24, 3: 0.17, 4: 0.08, 5: 0.05}

✅ Value Iteration hội tụ sau 180 vòng lặp.

📊 BẢNG GIÁ TRỊ TIỆN ÍCH (EXPECTED UTILITY) TẠI MỖI TRẠNG THÁI:
- Tồn kho 0 sản phẩm: 1885.41 (Giá trị kỳ vọng lâu dài)
- Tồn kho 1 sản phẩm: 1925.41 (Giá trị kỳ vọng lâu dài)
- Tồn kho 2 sản phẩm: 1965.41 (Giá trị kỳ vọng lâu dài)
- Tồn kho 3 sản phẩm: 2005.41 (Giá trị kỳ vọng lâu dài)
- Tồn kho 4 sản phẩm: 2045.41 (Giá trị kỳ vọng lâu dài)
- Tồn kho 5 sản phẩm: 2082.39 (Giá trị kỳ vọng lâu dài)

🚀 CHÍNH SÁCH NHẬP HÀNG TỐI ƯU (OPTIMAL POLICY π*):
--------------------------------------------------
Kho đang có [ 0 ] món  ->  Đặt nhập thêm [ 4 ] món.
Kho đang có [ 1 ] món  ->  Đặt nhập thêm [ 3 ] món.
Kho đang có [ 2 ] món  ->  Đặt nhập thêm [ 2 ] món.
Kho đang có [ 3 ] món  ->  Đặt nhập thêm [ 1 ] món.
Kho đang có [ 4 ] món  ->  Không cần nhập thêm.
Kho đang có [ 5 ] món  ->  Không cần nhập thêm.
